In [0]:
# ============================================================
# LOAD GLOBAL PROJECT CONFIGURATION
# ============================================================

# Loads reusable constants, API parameters,
# naming conventions and audit configuration
# shared across the project.

# MAGIC %run ./utils_config

In [0]:
%run ./utils_config

PROJECT CONFIGURATION LOADED SUCCESSFULLY
PROJECT_NAME: brazil_legislative_analytics
PROJECT_VERSION: v1.0.0
PROJECT_ENVIRONMENT: dev
CATALOG_NAME: brazil_legislative_analytics
RUN_ID: a926ff8a-70f4-467e-a2a3-77be77328083


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 99 Utils — Câmara API Client
# MAGIC
# MAGIC **Notebook:** `utils_api_client`
# MAGIC
# MAGIC Provides reusable functions to request data from the Câmara dos Deputados Open Data API.
# MAGIC
# MAGIC This notebook centralizes API access logic used by Bronze ingestion notebooks.
# MAGIC
# MAGIC ## Responsibilities
# MAGIC
# MAGIC - Execute HTTP GET requests against Câmara Open Data API
# MAGIC - Apply timeout and retry strategy
# MAGIC - Standardize API response handling
# MAGIC - Support resilient Bronze ingestion
# MAGIC - Preserve error details for auditability
# MAGIC
# MAGIC ## Naming Standard
# MAGIC
# MAGIC - Table and field names use Portuguese mnemonics.
# MAGIC - Comments and documentation are written in English.
# MAGIC - Utility functions use descriptive English names for reuse and readability.

# COMMAND ----------

import time
import requests

from typing import (
    Optional,
    Dict,
    Any,
    List,
)

# COMMAND ----------

# MAGIC %run ./utils_config

# COMMAND ----------

# ============================================================
# API CLIENT CONFIGURATION
# ============================================================

DEFAULT_REQUEST_HEADERS = {
    "Accept": "application/json",
    "User-Agent": (
        "brazil-legislative-analytics/"
        f"{PROJECT_VERSION}"
    ),
}

# COMMAND ----------

def build_request_url(
    endpoint_path: str,
) -> str:
    """
    Builds the full Câmara API request URL.
    """

    if endpoint_path is None:
        raise ValueError(
            "Endpoint path cannot be None."
        )

    normalized_endpoint = (
        str(endpoint_path)
        .strip()
    )

    if normalized_endpoint == "":
        raise ValueError(
            "Endpoint path cannot be empty."
        )

    if not normalized_endpoint.startswith("/"):
        normalized_endpoint = (
            f"/{normalized_endpoint}"
        )

    return (
        f"{CAMARA_API_BASE_URL}"
        f"{normalized_endpoint}"
    )

# COMMAND ----------

def parse_json_response(
    api_response: requests.Response,
    endpoint_path: str,
) -> Dict[str, Any]:
    """
    Parses an HTTP response as JSON.
    """

    try:
        parsed_response = (
            api_response.json()
        )

    except Exception as error:

        raise ValueError(
            f"Failed to parse API response as JSON "
            f"| endpoint={endpoint_path} "
            f"| status_code={api_response.status_code} "
            f"| error={str(error)}"
        )

    if not isinstance(parsed_response, dict):

        raise ValueError(
            f"API response is not a dictionary "
            f"| endpoint={endpoint_path}"
        )

    return parsed_response

# COMMAND ----------

def fetch_camara_api_data(
    endpoint_path: str,
    query_params: Optional[Dict[str, Any]] = None,
    request_timeout_seconds: int = API_REQUEST_TIMEOUT_SECONDS,
    max_retry_attempts: int = API_MAX_RETRY_ATTEMPTS,
    retry_sleep_seconds: int = API_RETRY_SLEEP_SECONDS,
    request_headers: Optional[Dict[str, str]] = None,
) -> Dict[str, Any]:
    """
    Requests data from a Câmara dos Deputados Open Data API endpoint.

    Parameters
    ----------
    endpoint_path : str
        API endpoint path, for example: "/deputados".

    query_params : dict, optional
        Query parameters sent to the API request.

    request_timeout_seconds : int
        Maximum request timeout in seconds.

    max_retry_attempts : int
        Maximum number of retry attempts.

    retry_sleep_seconds : int
        Waiting time between retry attempts.

    request_headers : dict, optional
        Additional request headers.

    Returns
    -------
    dict
        JSON response returned by the API.

    Raises
    ------
    RuntimeError
        Raised when all retry attempts fail.
    """

    request_url = build_request_url(
        endpoint_path=endpoint_path,
    )

    headers = (
        request_headers
        or DEFAULT_REQUEST_HEADERS
    )

    last_error = None

    for attempt_number in range(
        1,
        max_retry_attempts + 1,
    ):

        try:

            api_response = requests.get(
                request_url,
                params=query_params,
                timeout=request_timeout_seconds,
                headers=headers,
            )

            api_response.raise_for_status()

            return parse_json_response(
                api_response=api_response,
                endpoint_path=endpoint_path,
            )

        except requests.exceptions.Timeout as error:

            last_error = error
            error_type = "timeout"

        except requests.exceptions.HTTPError as error:

            last_error = error

            status_code = (
                error.response.status_code
                if error.response is not None
                else "unknown"
            )

            error_type = (
                f"http_error_{status_code}"
            )

        except requests.exceptions.RequestException as error:

            last_error = error
            error_type = "request_error"

        except Exception as error:

            last_error = error
            error_type = "unexpected_error"

        print(
            f"[WARNING] API request failed "
            f"| endpoint={endpoint_path} "
            f"| attempt={attempt_number}/{max_retry_attempts} "
            f"| error_type={error_type} "
            f"| error={str(last_error)}"
        )

        if (
            attempt_number
            < max_retry_attempts
        ):

            time.sleep(
                retry_sleep_seconds
                * attempt_number
            )

    raise RuntimeError(
        f"API request failed after "
        f"{max_retry_attempts} attempt(s) "
        f"| endpoint={endpoint_path} "
        f"| params={query_params} "
        f"| last_error={str(last_error)}"
    )

# COMMAND ----------

def extract_response_records(
    api_response: Dict[str, Any],
    data_field_name: str = API_RESPONSE_DATA_FIELD,
) -> List[Dict[str, Any]]:
    """
    Extracts the data records list from a Câmara API JSON response.

    Parameters
    ----------
    api_response : dict
        Full API JSON response.

    data_field_name : str
        Field name that contains the data records.

    Returns
    -------
    list
        List of records found in the API response.
    """

    if api_response is None:
        return []

    if not isinstance(api_response, dict):
        return []

    records = api_response.get(
        data_field_name,
        [],
    )

    if records is None:
        return []

    if not isinstance(records, list):
        return []

    return records

# COMMAND ----------

def check_endpoint_availability(
    endpoint_path: str,
    query_params: Optional[Dict[str, Any]] = None,
    request_timeout_seconds: int = 15,
    fail_silently: bool = True,
) -> bool:
    """
    Checks whether an API endpoint is available.

    Parameters
    ----------
    endpoint_path : str
        API endpoint path.

    query_params : dict, optional
        Query parameters sent to the API request.

    request_timeout_seconds : int
        Maximum timeout used for the availability check.

    fail_silently : bool
        When True, returns False instead of raising exceptions.

    Returns
    -------
    bool
        True when the endpoint returns a valid response,
        otherwise False.
    """

    try:

        fetch_camara_api_data(
            endpoint_path=endpoint_path,
            query_params=query_params,
            request_timeout_seconds=request_timeout_seconds,
            max_retry_attempts=1,
        )

        return True

    except Exception as error:

        if not fail_silently:
            raise error

        print(
            f"[WARNING] Endpoint availability check failed "
            f"| endpoint={endpoint_path} "
            f"| error={str(error)}"
        )

        return False

# COMMAND ----------

print("utils_api_client loaded successfully.")

utils_api_client loaded successfully.


utils_config loaded successfully.
